# Accumulation of position errors in internally drifting stimuli
**Saleki, Cavanagh & Tse — analysis notebook for manuscript v16**

This notebook reproduces every number and every data figure of the paper from the trial files in `../data/`:

1. rebuilds the two preprocessed tables from the raw trial files and checks that they are identical to `../data/preproc/`;
2. orientation adjustment task: illusory angle by path duration, repeated-measures ANOVA, individual slopes (Figure 3a);
3. length matching task: illusory offsets of the single-drift and double-drift (Eq. 1), two-way repeated-measures ANOVA, paired tests (Figure 3b);
4. participant-by-participant comparison (Figure 4);
5. comparison with the literature in envelope-σ units (Figure 5), with this study's values taken from the computations above.

Analysis conventions (as in the Methods): every orientation setting counts as a report; the only exclusion is the 3-standard-deviation screen within participant and condition; statistics are computed on participant means (six participants) with Greenhouse–Geisser (GG) correction for repeated-measures ANOVAs.

Figures are saved to `notebook_output/`. Requirements: `pandas numpy scipy matplotlib pingouin statsmodels`.

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
from scipy import stats
import pingouin as pg
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")
%matplotlib inline

HERE = os.getcwd()                                   # analysis/
DATA = os.path.join(HERE, "..", "data")
OUT = os.path.join(HERE, "notebook_output"); os.makedirs(OUT, exist_ok=True)

SIGMA = 0.3                                          # envelope standard deviation (dva): PsychoPy gauss mask, size 1.8 dva = ±3σ
DURS = [50, 200, 350, 500, 650, 800]
PHYS = {50: 0.125, 200: 0.5, 350: 0.875, 500: 1.25, 650: 1.625, 800: 2.0}   # physical path length (dva) at 2.5 dva/s
SUBS = [1, 3, 4, 5, 7, 8]
SUB_COLORS = {1: "#E800B6", 3: "#7b5740", 4: "#00BA09", 5: "#00D19D", 7: "#00FFFF", 8: "#0099FF"}
LABELS = {s: f"Participant {i+1}" for i, s in enumerate(SUBS)}
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 15, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.labelsize": 17, "xtick.labelsize": 14, "ytick.labelsize": 14, "axes.linewidth": 1.2})
R = {}                                               # every reported number is collected here
def show(df, title=None, fmt="{:.3f}"):
    if title: display(Markdown(f"**{title}**"))
    if isinstance(df, pd.DataFrame):
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c])]   # format numbers only
        display(df.style.format(fmt, subset=pd.IndexSlice[:, num_cols], na_rep="") if num_cols else df)
    else: display(df)

## 1. Data: rebuild the preprocessed tables from the raw trial files
The experiment code labelled the orientation adjustment task `match` and the length matching task `adjustment`; the preprocessed tables add `ANGLE_FROM_VERTICAL` (90° minus the grating orientation setting, or minus the double-drift's path orientation) and rename the task.

In [ ]:
import glob
raw = sorted(glob.glob(os.path.join(DATA, "raw", "*.csv")))
ori_raw = pd.concat([pd.read_csv(f) for f in raw if "task-match" in f], ignore_index=True)        # orientation adjustment task
len_raw = pd.concat([pd.read_csv(f) for f in raw if "task-adjustment" in f], ignore_index=True)   # length matching task
ori_raw["ANGLE_FROM_VERTICAL"] = 90 - ori_raw["ANGLE"]
len_raw["ANGLE_FROM_VERTICAL"] = 90 - len_raw["ENVELOPE_ANGLE"]
ori_raw["TASK"] = "ORI"; len_raw["TASK"] = "LENGTH"                  # the preprocessed tables rename the tasks
ori_pre = pd.read_csv(os.path.join(DATA, "preproc", "speed-2.5_df-ori.csv"))
len_pre = pd.read_csv(os.path.join(DATA, "preproc", "speed-2.5_df-len.csv"))

def same(a, b, keys):
    cols = [c for c in b.columns if c in a.columns]
    a2 = a.sort_values(keys)[cols].reset_index(drop=True); b2 = b.sort_values(keys)[cols].reset_index(drop=True)
    num = a2.select_dtypes("number").columns
    ok = a2[num].sub(b2[num]).abs().max().max() < 1e-9 and (a2.drop(columns=num) == b2.drop(columns=num)).all().all()
    return ok, len(cols)
ok1, n1 = same(ori_raw, ori_pre, ["SUB_ID", "BLOCK", "TRIAL"]); ok2, n2 = same(len_raw, len_pre, ["SUB_ID", "BLOCK", "TRIAL"])
print(f"orientation task: {len(ori_pre)} trials, {ori_pre.SUB_ID.nunique()} participants; rebuilt table identical on {n1} shared columns: {ok1}")
print(f"length task:      {len(len_pre)} trials, {len_pre.SUB_ID.nunique()} participants; rebuilt table identical on {n2} shared columns: {ok2}")
assert ok1 and ok2
ori, ln = ori_pre.copy(), len_pre.copy()
show(ori.groupby(["SUB_ID", "DURATION_MS"]).size().unstack(), "Orientation trials per participant and path duration", "{:d}")
show(ln.groupby(["SUB_ID", "STIM_TYPE", "DURATION_MS"]).size().unstack(), "Length trials per participant, stimulus type and path duration", "{:d}")

## 2. Orientation adjustment task: the illusory angle
A setting equal to the initial 45° means the participant pressed the key without turning the wheel; these settings are kept as reports (Methods, v16). The only exclusion is the 3-SD screen within participant and path duration. The angle is analysed on participant means with a one-way repeated-measures ANOVA (GG-corrected) and by testing the participants' individual slopes against zero.

In [ ]:
def z_within(df, val, by):
    g = df.groupby(by)[val]; return (df[val] - g.transform("mean")) / g.transform("std")
ori["Z"] = z_within(ori, "ANGLE_FROM_VERTICAL", ["SUB_ID", "DURATION_MS"])
oa = ori[~(ori["Z"].abs() > 3)].copy()                                   # "more than 3 SD" excluded; a NaN z (zero-variance cell) is kept
R["ori_trials_before"], R["ori_trials_after"] = len(ori), len(oa)
R["settings_at_45_total"] = int((ori.ANGLE == 45).sum()); R["settings_at_45_kept"] = int((oa.ANGLE == 45).sum())
print(f"settings: {len(ori)} → {len(oa)} after the 3-SD screen ({len(ori)-len(oa)} removed); settings left at 45°: {R['settings_at_45_total']} ({R['settings_at_45_kept']} kept)")

ang = oa.groupby(["SUB_ID", "DURATION_MS"])["ANGLE_FROM_VERTICAL"].mean().unstack()
R["angle_subject_means"] = ang.round(2).to_dict()
R["angle_group_mean_by_duration"] = ang.mean().to_dict(); R["angle_group_sem_by_duration"] = ang.sem().to_dict()
R["angle_grand_mean"] = float(ang.mean(axis=1).mean()); R["angle_subject_grand_means"] = ang.mean(axis=1).to_dict()
R["angle_within_cell_sd_median"] = float(oa.groupby(["SUB_ID", "DURATION_MS"])["ANGLE_FROM_VERTICAL"].std().median())
show(ang.assign(mean=ang.mean(axis=1)).round(1), "Illusory angle (°): participant means by path duration", "{:.1f}")
print(f"group means by duration: {', '.join(f'{v:.1f}°' for v in ang.mean())}; grand mean {R['angle_grand_mean']:.1f}°; participant means {ang.mean(axis=1).min():.1f}° to {ang.mean(axis=1).max():.1f}°")
print(f"median within-condition SD of the settings: {R['angle_within_cell_sd_median']:.1f}°")

ang_long = ang.stack().rename("ANGLE").reset_index()
ra = pg.rm_anova(data=ang_long, dv="ANGLE", within="DURATION_MS", subject="SUB_ID", detailed=True, correction=True)
df1 = int(ra["ddof1"][0]) if "ddof1" in ra else int(ra["DF"][0]); df2 = int(ra["ddof2"][0]) if "ddof2" in ra else int(ra["DF"][1])   # pingouin versions differ
R["angle_rm_anova"] = dict(F=float(ra["F"][0]), df1=df1, df2=df2, p_unc=float(ra["p_unc"][0]), p_GG=float(ra["p_GG_corr"][0]), eps=float(ra["eps"][0]))
show(ra, "One-way repeated-measures ANOVA on the illusory angle (GG-corrected p in p_GG_corr)", "{:.4g}")
slopes = oa.groupby("SUB_ID").apply(lambda g: np.polyfit(g["DURATION_MS"] / 1000, g["ANGLE_FROM_VERTICAL"], 1)[0])   # deg per s
t = stats.ttest_1samp(slopes, 0)
R["angle_slopes_deg_per_s"] = slopes.to_dict(); R["angle_slope_mean"] = float(slopes.mean()); R["angle_slope_t"] = float(t.statistic); R["angle_slope_p"] = float(t.pvalue)
print(f"individual slopes (°/s): {', '.join(f'{v:.1f}' for v in slopes)}; mean {slopes.mean():.1f}°/s, t({len(slopes)-1}) = {t.statistic:.2f}, p = {t.pvalue:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.2))
for s in SUBS:
    g = oa[oa.SUB_ID == s].groupby("DURATION_MS")["ANGLE_FROM_VERTICAL"]
    ax.errorbar(DURS, g.mean().reindex(DURS), yerr=g.sem().reindex(DURS), color=SUB_COLORS[s], alpha=0.45, lw=1.6, marker="o", ms=4, capsize=0)
ax.errorbar(DURS, [ang[d].mean() for d in DURS], yerr=[ang[d].sem() for d in DURS], color="#be39d6", lw=3.5, marker="o", ms=9, capsize=0, zorder=5)
ax.set_xticks(DURS); ax.set_ylim(40, 90); ax.set_yticks([40, 50, 60, 70, 80, 90]); ax.set_xlabel("Path duration (ms)"); ax.set_ylabel("Illusory angle (°)")
ax.set_title("Figure 3a", loc="left", fontsize=13); fig.tight_layout(); plt.show()

## 3. Length matching task: illusory offsets (Eq. 1)
For the single-drift the illusory offset is the matched length of the perceived path, the displacement accumulated during one pass of unidirectional internal drift. For the double-drift it is the component of the perceived path in the direction of the internal motion, $L_i = L \sin\theta$, with $L$ the matched length and $\theta$ the illusory angle (the orientation setting used for that participant, path duration and hemifield). Reports more than 3 SD from the mean of their participant, stimulus type and duration are excluded.

In [ ]:
ln["Z"] = z_within(ln, "REPORT_LENGTH_DVA", ["SUB_ID", "STIM_TYPE", "DURATION_MS"])
lc = ln[~(ln["Z"].abs() > 3)].copy()                                      # same rule; one cell (participant 7, double-drift, 50 ms) has 24 identical reports of 0.30 dva and is kept
R["len_trials_before"], R["len_trials_after"] = len(ln), len(lc)
print(f"reports: {len(ln)} → {len(lc)} after the 3-SD screen ({len(ln)-len(lc)} removed); zero-variance cells kept: {int((ln.groupby(['SUB_ID', 'STIM_TYPE', 'DURATION_MS'])['REPORT_LENGTH_DVA'].std() == 0).sum())}")
lc["OFFSET"] = np.where(lc["STIM_TYPE"] == "dd", lc["REPORT_LENGTH_DVA"] * np.sin(np.deg2rad(lc["ANGLE_FROM_VERTICAL"])), lc["REPORT_LENGTH_DVA"])
cell = lc.groupby(["SUB_ID", "STIM_TYPE", "DURATION_MS"]).agg(OFFSET=("OFFSET", "mean"), RAW=("REPORT_LENGTH_DVA", "mean"), N=("OFFSET", "size")).reset_index()
cell.to_csv(os.path.join(OUT, "cell_means.csv"), index=False)
grp = cell.groupby(["STIM_TYPE", "DURATION_MS"])["OFFSET"].agg(["mean", "sem"])
tbl = pd.DataFrame({("single-drift", "mean"): [grp.loc[("deva", d), "mean"] for d in DURS], ("single-drift", "SEM"): [grp.loc[("deva", d), "sem"] for d in DURS],
                    ("single-drift", "σ"): [grp.loc[("deva", d), "mean"] / SIGMA for d in DURS],
                    ("double-drift", "mean"): [grp.loc[("dd", d), "mean"] for d in DURS], ("double-drift", "SEM"): [grp.loc[("dd", d), "sem"] for d in DURS],
                    ("double-drift", "σ"): [grp.loc[("dd", d), "mean"] / SIGMA for d in DURS]}, index=pd.Index(DURS, name="path duration (ms)"))
show(tbl, "Illusory offset (dva, and in envelope σ): group means ± SEM across the six participants", "{:.3f}")
R["offset_group_means"] = {st: {int(d): dict(mean=float(grp.loc[(st, d), "mean"]), sem=float(grp.loc[(st, d), "sem"])) for d in DURS} for st in ["deva", "dd"]}
sd800 = cell[(cell.STIM_TYPE == "deva") & (cell.DURATION_MS == 800)].set_index("SUB_ID")["OFFSET"]; dd800 = cell[(cell.STIM_TYPE == "dd") & (cell.DURATION_MS == 800)].set_index("SUB_ID")["OFFSET"]
R["sd_800_by_subject"] = sd800.to_dict(); R["dd_800_by_subject"] = dd800.to_dict()
R["dd_raw_length_over_physical"] = {int(d): float(cell[(cell.STIM_TYPE == "dd") & (cell.DURATION_MS == d)]["RAW"].mean() / PHYS[d]) for d in DURS}
print(f"single-drift at 800 ms across participants: {sd800.min():.2f} to {sd800.max():.2f} dva; double-drift: {dd800.min():.2f} to {dd800.max():.2f} dva")
print("double-drift matched length / physical path length by duration: " + ", ".join(f"{d} ms: {v:.2f}" for d, v in R["dd_raw_length_over_physical"].items()))

In [ ]:
r2 = pg.rm_anova(data=cell, dv="OFFSET", within=["STIM_TYPE", "DURATION_MS"], subject="SUB_ID", detailed=True, correction=True)
R["offset_rm_anova"] = {str(r2["Source"][i]): dict(F=float(r2["F"][i]), df1=int(r2["ddof1"][i]), df2=int(r2["ddof2"][i]), p_unc=float(r2["p_unc"][i]), p_GG=float(r2["p_GG_corr"][i])) for i in range(len(r2))}
show(r2, "Two-way repeated-measures ANOVA (stimulus type × path duration) on the illusory offset, six participants", "{:.4g}")
wide = cell.pivot_table(index=["SUB_ID", "DURATION_MS"], columns="STIM_TYPE", values="OFFSET").reset_index()
sm = wide.groupby("SUB_ID")[["deva", "dd"]].mean()
tt = pg.ttest(sm["dd"], sm["deva"], paired=True)
R["diff_mean"] = float((sm["dd"] - sm["deva"]).mean()); R["diff_pct_of_grand_mean"] = float(100 * (sm["dd"].mean() / sm["deva"].mean() - 1))
R["diff_t"], R["diff_dof"], R["diff_p"] = float(tt["T"].iloc[0]), int(tt["dof"].iloc[0]), float(tt["p_val"].iloc[0])
print(f"double-drift minus single-drift (participant means): {R['diff_mean']:.3f} dva ({R['diff_pct_of_grand_mean']:.1f}% of the single-drift grand mean); paired t({R['diff_dof']}) = {R['diff_t']:.2f}, p = {R['diff_p']:.3f}")
def pt(a, b):
    t = pg.ttest(a, b, paired=True); return dict(t=float(t["T"].iloc[0]), dof=int(t["dof"].iloc[0]), p=float(t["p_val"].iloc[0]))
per = {}
for d in DURS:
    a = cell[(cell.STIM_TYPE == "dd") & (cell.DURATION_MS == d)].set_index("SUB_ID")["OFFSET"]; b = cell[(cell.STIM_TYPE == "deva") & (cell.DURATION_MS == d)].set_index("SUB_ID")["OFFSET"]
    per[d] = dict(mean_diff=float((a - b).mean()), **pt(a, b))
R["diff_by_duration"] = per
show(pd.DataFrame(per).T.rename_axis("path duration (ms)"), "Double-drift minus single-drift at each path duration (paired t-tests)", "{:.3f}")
wsd = cell[cell.STIM_TYPE == "deva"].pivot(index="SUB_ID", columns="DURATION_MS", values="OFFSET"); wdd = cell[cell.STIM_TYPE == "dd"].pivot(index="SUB_ID", columns="DURATION_MS", values="OFFSET")
R["growth"] = {"single 800 vs 500": pt(wsd[800], wsd[500]), "single 800 vs 650": pt(wsd[800], wsd[650]), "double 800 vs 500": pt(wdd[800], wdd[500]), "double 800 vs 650": pt(wdd[800], wdd[650])}
show(pd.DataFrame(R["growth"]).T, "Growth at the longest durations (paired t-tests on participant means)", "{:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.2))
for s in SUBS:
    for st, ls in [("deva", "-"), ("dd", "--")]:
        g = cell[(cell.SUB_ID == s) & (cell.STIM_TYPE == st)].set_index("DURATION_MS").reindex(DURS)
        ax.plot(DURS, g["OFFSET"], ls=ls, color=SUB_COLORS[s], alpha=0.4, lw=1.4, marker="o", ms=3.5)
for st, ls, lab in [("deva", "-", "Single-drift"), ("dd", "--", "Double-drift")]:
    ax.errorbar(DURS, [grp.loc[(st, d), "mean"] for d in DURS], yerr=[grp.loc[(st, d), "sem"] for d in DURS], color="#FF142C", ls=ls, lw=3.5, marker="o", ms=9, capsize=0, zorder=5, label=lab)
ax.set_xticks(DURS); ax.set_ylim(0, 2.0); ax.set_xlabel("Path duration (ms)"); ax.set_ylabel("Illusory offset (dva)")
sec = ax.secondary_yaxis("right", functions=(lambda v: v / SIGMA, lambda v: v * SIGMA)); sec.set_ylabel("Illusory offset (σ)"); sec.spines["right"].set_visible(True)
leg = ax.legend(frameon=False, loc="upper left", fontsize=13, handlelength=3.4, handletextpad=0.8)
for lh in leg.get_lines(): lh.set_linewidth(2.2); lh.set_markersize(6)
ax.set_title("Figure 3b", loc="left", fontsize=13); fig.tight_layout(); plt.show()

# composite Figure 3 as in the manuscript
fig, axes = plt.subplots(1, 2, figsize=(12.4, 5.4))
for a_, lab in zip(axes, ["(a)", "(b)"]): a_.text(-0.18, 1.04, lab, transform=a_.transAxes, fontsize=22)
ax = axes[0]
for s in SUBS:
    g = oa[oa.SUB_ID == s].groupby("DURATION_MS")["ANGLE_FROM_VERTICAL"]
    ax.errorbar(DURS, g.mean().reindex(DURS), yerr=g.sem().reindex(DURS), color=SUB_COLORS[s], alpha=0.45, lw=1.6, marker="o", ms=4, capsize=0)
ax.errorbar(DURS, [ang[d].mean() for d in DURS], yerr=[ang[d].sem() for d in DURS], color="#be39d6", lw=3.5, marker="o", ms=9, capsize=0, zorder=5)
ax.set_xticks(DURS); ax.set_ylim(40, 90); ax.set_yticks([40, 50, 60, 70, 80, 90]); ax.set_xlabel("Path duration (ms)"); ax.set_ylabel("Illusory angle (°)")
ax = axes[1]
for s in SUBS:
    for st, ls in [("deva", "-"), ("dd", "--")]:
        g = cell[(cell.SUB_ID == s) & (cell.STIM_TYPE == st)].set_index("DURATION_MS").reindex(DURS)
        ax.plot(DURS, g["OFFSET"], ls=ls, color=SUB_COLORS[s], alpha=0.4, lw=1.4, marker="o", ms=3.5)
for st, ls, lab in [("deva", "-", "Single-drift"), ("dd", "--", "Double-drift")]:
    ax.errorbar(DURS, [grp.loc[(st, d), "mean"] for d in DURS], yerr=[grp.loc[(st, d), "sem"] for d in DURS], color="#FF142C", ls=ls, lw=3.5, marker="o", ms=9, capsize=0, zorder=5, label=lab)
ax.set_xticks(DURS); ax.set_ylim(0, 2.0); ax.set_xlabel("Path duration (ms)"); ax.set_ylabel("Illusory offset (dva)")
sec = ax.secondary_yaxis("right", functions=(lambda v: v / SIGMA, lambda v: v * SIGMA)); sec.set_ylabel("Illusory offset (σ)"); sec.spines["right"].set_visible(True)
leg = ax.legend(frameon=False, loc="upper left", fontsize=13, handlelength=3.4, handletextpad=0.8)
for lh in leg.get_lines(): lh.set_linewidth(2.2); lh.set_markersize(6)
fig.tight_layout(w_pad=3); fig.savefig(os.path.join(OUT, "Figure3.png"), dpi=300); fig.savefig(os.path.join(OUT, "Figure3.pdf")); plt.close(fig)
print("saved notebook_output/Figure3.png and .pdf")

## 4. Participant-by-participant comparison (Figure 4)
If the two offsets index the same process they should covary across participants, not merely agree on average.

In [ ]:
r, p = stats.pearsonr(sm["deva"], sm["dd"]); lr = stats.linregress(sm["deva"], sm["dd"]); rho, prho = stats.spearmanr(sm["deva"], sm["dd"])
ratios = sm["dd"] / sm["deva"]
R["corr"] = dict(r=float(r), p=float(p), slope=float(lr.slope), intercept=float(lr.intercept), spearman_rho=float(rho), spearman_p=float(prho), ratio_min=float(ratios.min()), ratio_max=float(ratios.max()))
R["corr_per_duration"] = {int(d): dict(r=float(stats.pearsonr(g["deva"], g["dd"])[0]), p=float(stats.pearsonr(g["deva"], g["dd"])[1])) for d, g in wide.groupby("DURATION_MS")}
show(sm.assign(ratio=ratios).rename(columns={"deva": "single-drift", "dd": "double-drift"}), "Participant means over path durations (dva) and their ratio", "{:.3f}")
print(f"Pearson r = {r:.3f} (p = {p:.4f}); Spearman ρ = {rho:.3f} (p = {prho:.4f}); slope {lr.slope:.3f}, intercept {lr.intercept:.3f} dva; ratios {ratios.min():.2f} to {ratios.max():.2f}")
show(pd.DataFrame(R["corr_per_duration"]).T.rename_axis("path duration (ms)"), "Between-participant correlation at each path duration", "{:.4f}")
fig, ax = plt.subplots(figsize=(6.4, 6.0)); lim2 = (0.4, 1.2)
ax.plot(lim2, lim2, ls=":", color="#777777", lw=1.5, zorder=1)
for s in sm.index: ax.scatter(sm.loc[s, "deva"], sm.loc[s, "dd"], s=360, color=SUB_COLORS[s], edgecolor="white", lw=1, zorder=3)
ax.text(0.04, 0.95, f"r = {r:.2f}, p = {p:.3f}", transform=ax.transAxes, va="top", fontsize=14)
ax.set_xlim(lim2); ax.set_ylim(lim2); ax.set_aspect("equal"); ax.set_xlabel("Single-drift offset (dva)"); ax.set_ylabel("Double-drift offset (dva)")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "Figure4.png"), dpi=300); fig.savefig(os.path.join(OUT, "Figure4.pdf")); plt.show()

## 5. Comparison with the literature (Figure 5)
Values for the cited studies were read from their text or figures (`figure5_points_v16.csv`, `figure5_tf_points_v16.csv`, with the source noted per row). Two-patch (relative) measurements are halved to give the shift of one patch. This study's points are replaced by the values computed above: the single-drift and double-drift offsets at 800 ms in σ, with the range across participants.

In [ ]:
pts = pd.read_csv(os.path.join(DATA, "literature", "figure5_points_v16.csv")); tfp = pd.read_csv(os.path.join(DATA, "literature", "figure5_tf_points_v16.csv"))
ours = {"ours_sd": (sd800.mean() / SIGMA, sd800.min() / SIGMA, sd800.max() / SIGMA), "ours_dd": (dd800.mean() / SIGMA, dd800.min() / SIGMA, dd800.max() / SIGMA)}
for k, (m, lo, hi) in ours.items():
    pts.loc[pts.kind == k, ["shift_sigma_one_patch", "range_lo", "range_hi"]] = [m, lo, hi]; tfp.loc[tfp.kind == k, "shift_sigma_one_patch"] = m
R["fig5_ours_sigma"] = {k: dict(mean=v[0], lo=v[1], hi=v[2]) for k, v in ours.items()}
print("this study in σ:", {k: f"{v[0]:.2f} (range {v[1]:.2f} to {v[2]:.2f})" for k, v in ours.items()})
show(pts[pts.kind != "omitted"][["study", "ecc_dva", "shift_sigma_one_patch", "kind", "source"]].reset_index(drop=True), "Figure 5a points (one-patch shift in σ; two-patch studies halved)", "{:.2f}")

red, grey, purple = "#FF142C", "#3F3F3F", "#be39d6"
with plt.rc_context({"font.size": 13, "axes.labelsize": 15, "xtick.labelsize": 12, "ytick.labelsize": 12, "axes.linewidth": 1.1}):
    def style_point(ax, x, y, kind, lo=None, hi=None, xbar=0.18):
        if kind == "one": ax.scatter(x, y, s=70, color=grey, zorder=3)
        elif kind == "rel": ax.scatter(x, y, s=70, facecolor="white", edgecolor=grey, lw=1.6, zorder=3)
        elif kind in ("ours_sd", "ours_dd"):
            if lo is not None:
                ax.plot([x] * 2, [lo, hi], color=red, lw=1.8, zorder=4); ax.plot([x - xbar, x + xbar], [lo] * 2, color=red, lw=1.4, zorder=4); ax.plot([x - xbar, x + xbar], [hi] * 2, color=red, lw=1.4, zorder=4)
            if kind == "ours_sd": ax.scatter(x, y, s=120, marker="D", color=red, zorder=5)
            else: ax.scatter(x, y, s=120, marker="D", facecolor="white", edgecolor=red, lw=2, zorder=5)
        elif kind == "dd_lit": ax.scatter(x, y, s=130, marker="*", color=purple, zorder=4)
    def label(ax, text, x, y, off, ha="left"): ax.annotate(text, (x, y), xytext=off, textcoords="offset points", fontsize=9.5, color=grey, ha=ha, va="center")
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8), sharey=True, gridspec_kw={"width_ratios": [1.15, 1]})
    # (a) eccentricity
    ax = axes[0]
    dv = pts[pts.study.str.startswith("De Valois")]; ax.plot(dv.ecc_dva, dv.shift_sigma_one_patch, "-", color=grey, lw=1.5, zorder=2)
    for _, row in pts.iterrows():
        if row.kind == "omitted": continue
        style_point(ax, row.ecc_dva + (0.55 if row.kind == "ours_dd" else 0.0), row.shift_sigma_one_patch, row.kind, row.get("range_lo"), row.get("range_hi"))
    for text, (x, y, off, ha) in {"De Valois & De Valois (1991)": (8.0, 1.88, (-8, 12), "right"), "Chung et al. (2007)": (4.0, 0.347, (7, 3), "left"), "Bressler & Whitney (2006)": (9.1, 0.35, (7, 3), "left"),
                                  "Kosovicheva et al. (2014)": (10.0, 0.846, (7, 2), "left"), "Ye et al. (2018)": (3.0, 0.30, (6, -12), "left"), "Hu et al. (2022)": (4.0, 0.5, (6, 9), "left")}.items():
        label(ax, text, x, y, off, ha)
    ax.set_yscale("log"); ax.set_ylim(0.08, 40); ax.set_xlim(-0.8, 14); ax.set_yticks([0.1, 0.3, 1, 3, 10, 30]); ax.set_yticklabels(["0.1", "0.3", "1", "3", "10", "30"]); ax.set_xticks([0, 2, 4, 6, 8, 10, 12, 14])
    ax.set_xlabel("Eccentricity (dva)"); ax.set_ylabel("Position shift of one patch (envelope σ)")
    h = [Line2D([], [], marker="o", color=grey, lw=0, ms=8, label="single patch, one-sided"), Line2D([], [], marker="o", mfc="white", mec=grey, color=grey, lw=0, ms=8, label="two patches, halved"),
         Line2D([], [], marker="D", color=red, lw=0, ms=8, label="this study, single-drift"), Line2D([], [], marker="D", mfc="white", mec=red, color=red, lw=0, ms=8, label="this study, double-drift"),
         Line2D([], [], marker="*", color=purple, lw=0, ms=11, label="Lisi & Cavanagh (2015), double-drift")]
    ax.legend(handles=h, loc="upper left", fontsize=9.5, frameon=False); ax.text(-0.14, 1.02, "(a)", transform=ax.transAxes, fontsize=20)
    # (b) temporal frequency
    ax = axes[1]
    Tb = tfp[~tfp.series.str.contains("SF series") & ~(tfp.series == "TF series at 2 deg") & ~(tfp.series == "4 c/deg, 4 deg, 453 ms")]
    for (study, ser), ls in {("Chung et al. 2007", "1 c/deg, 4 deg, 453 ms"): "--", ("Bressler & Whitney 2006", "0.18 c/deg, 9 deg"): ":"}.items():
        g = Tb[(Tb.study == study) & (Tb.series == ser)].sort_values("tf_hz"); ax.plot(g.tf_hz, g.shift_sigma_one_patch, ls, color=grey, lw=1.6, zorder=2)
    rng = {k: (v[1], v[2]) for k, v in ours.items()}
    for _, row in Tb.iterrows():
        x = row.tf_hz * (1.12 if row.kind == "ours_dd" else 1.0)
        if row.kind in rng:
            lo, hi = rng[row.kind]; ax.plot([x] * 2, [lo, hi], color=red, lw=1.8, zorder=4); ax.plot([x * 0.95, x * 1.05], [lo] * 2, color=red, lw=1.4, zorder=4); ax.plot([x * 0.95, x * 1.05], [hi] * 2, color=red, lw=1.4, zorder=4)
        style_point(ax, x, row.shift_sigma_one_patch, row.kind)
    for text, (x, y, off, ha) in {"De Valois, 8°": (4.0, 1.88, (-8, 0), "right"), "De Valois, 1–2°": (4.0, 0.63, (-8, 2), "right"), "De Valois, 0°": (4.0, 0.25, (-8, -10), "right"), "Kosovicheva, 10°": (4.0, 0.846, (-8, 4), "right"),
                                  "Hu, 4°": (4.68, 0.5, (7, 7), "left"), "Ye, 9°": (15.0, 0.5, (7, 6), "left"), "Ye, 3°": (15.0, 0.30, (9, 9), "left"), "Chung, 4°": (1.0, 0.32, (0, 11), "center"), "Bressler & Whitney, 9°": (25.0, 0.229, (7, -7), "left")}.items():
        label(ax, text, x, y, off, ha)
    ax.set_xscale("log"); ax.set_xlim(0.15, 60); ax.set_xticks([0.3, 1, 3, 10, 30]); ax.set_xticklabels(["0.3", "1", "3", "10", "30"]); ax.set_xlabel("Temporal frequency of the internal motion (Hz)")
    ax.text(-0.08, 1.02, "(b)", transform=ax.transAxes, fontsize=20)
    fig.tight_layout(w_pad=2.5); fig.savefig(os.path.join(OUT, "Figure5.png"), dpi=300); fig.savefig(os.path.join(OUT, "Figure5.pdf")); plt.show()
json.dump(R, open(os.path.join(OUT, "results_notebook.json"), "w"), indent=2, default=float)
print("results saved to notebook_output/results_notebook.json")